# Paper QA With Section Attributions

Given a paper id and a user question, this notebook retrieves paper-scoped KG evidence from Fuseki and returns:

`question`, `answer`, and `attribute_uris`.

The retrieval path reuses the URI resolver's Fuseki helpers and the existing Jena text index. Section URIs are used as the primary attribution target.

## Setup

This expects the repo `.env` to contain `FUSEKI_SERVER_URL` and `FUSEKI_DATASET`. A local paper id such as `learning_relation_entailment_with_structured_and_textual_information` is expanded to `https://w3id.org/twc/sudo/kg/paper/{paper_id}` unless you pass a full URI.

In [1]:
from __future__ import annotations

import json
import logging
import math
import os
import re
from typing import Any
from urllib.parse import quote, urlsplit

logging.disable(logging.INFO)
from uri_resolver.backend import FusekiRedirectBackend
from uri_resolver.main import (
    _fetch_select_bindings_from_datasets,
    _ordered_dataset_names,
    fetch_fuseki_dataset_names,
)
from uri_resolver.settings import AppSettings
logging.disable(logging.NOTSET)


In [2]:
settings = AppSettings()
logging.getLogger("uri_resolver").setLevel(logging.WARNING)
logging.getLogger("uri_resolver.doc").setLevel(logging.WARNING)
logging.getLogger("uri_resolver.fuseki").setLevel(logging.WARNING)

backend = FusekiRedirectBackend(
    server_url=settings.fuseki_server_url,
    dataset=settings.fuseki_dataset,
)

dataset_names = _ordered_dataset_names(
    settings.fuseki_dataset,
    fetch_fuseki_dataset_names(settings.fuseki_server_url),
)
persistent_uri_base = os.getenv(
    "PAPER_QA_PERSISTENT_URI_BASE",
    settings.persistent_uri_base,
).rstrip("/")

{
    "fuseki_server_url": settings.fuseki_server_url,
    "default_dataset": backend.default_dataset,
    "dataset_search_order": dataset_names,
    "persistent_uri_base": persistent_uri_base,
}


{'fuseki_server_url': 'http://sudo-kg.idea.rpi.edu:3030',
 'default_dataset': 'gold_standard_kg',
 'dataset_search_order': ['gold_standard_kg'],
 'persistent_uri_base': 'https://w3id.org/twc/sudo/kg'}

## Retrieval Helpers

The KG stores provenance, labels, and structure across named graphs, so the queries intentionally join across multiple `GRAPH ?g { ... }` blocks instead of requiring all triples to live in one named graph.

In [3]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "how", "in",
    "into", "is", "it", "of", "on", "or", "paper", "that", "the", "their", "this",
    "to", "was", "were", "what", "when", "where", "which", "who", "why", "with",
}


def paper_uri_from_id(paper_id: str) -> str:
    value = paper_id.strip()
    if not value:
        raise ValueError("paper_id must be non-empty")
    if value.startswith("http://") or value.startswith("https://"):
        return value
    return f"{persistent_uri_base}/paper/{quote(value.strip('/'), safe='-._~')}"


def sparql_iri(uri: str) -> str:
    parsed = urlsplit(uri)
    if parsed.scheme not in {"http", "https", "urn"}:
        raise ValueError(f"Unsupported URI for SPARQL IRI: {uri}")
    if any(char in uri for char in "<>\n\r"):
        raise ValueError(f"Unsafe URI for SPARQL IRI: {uri}")
    return f"<{uri}>"


def sparql_string(value: str) -> str:
    return json.dumps(value)


def binding_value(binding: dict[str, Any], name: str) -> str | None:
    cell = binding.get(name)
    if not isinstance(cell, dict):
        return None
    value = cell.get("value")
    return value if isinstance(value, str) else None


def binding_float(binding: dict[str, Any], name: str) -> float | None:
    value = binding_value(binding, name)
    if value is None:
        return None
    try:
        return float(value)
    except ValueError:
        return None


def binding_int(binding: dict[str, Any], name: str) -> int | None:
    value = binding_value(binding, name)
    if value is None:
        return None
    try:
        return int(value)
    except ValueError:
        return None


def question_terms(question: str) -> list[str]:
    terms = re.findall(r"[A-Za-z0-9][A-Za-z0-9_-]+", question.lower())
    return [term for term in terms if len(term) > 2 and term not in STOPWORDS]


def lucene_query_from_question(question: str, max_terms: int = 18) -> str:
    terms = question_terms(question)
    if not terms:
        terms = re.findall(r"[A-Za-z0-9]+", question.lower())
    return " ".join(terms[:max_terms])


def run_select(query: str) -> list[dict[str, Any]]:
    return _fetch_select_bindings_from_datasets(
        backend=backend,
        resource_label="paper_qa",
        query=query,
        dataset_names=dataset_names,
    )


In [4]:
POSITION = "https://w3id.org/twc/sudo/kg/position"


def paper_metadata_query(paper_uri: str) -> str:
    paper = sparql_iri(paper_uri)
    return f"""
PREFIX po: <http://www.essepuntato.it/2008/12/pattern#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?predicate ?value ?displayValue WHERE {{
  GRAPH ?metadataGraph {{
    {paper} ?predicate ?value .
    FILTER(?predicate != po:contains)
  }}
  OPTIONAL {{ GRAPH ?valueLabelGraph {{ ?value rdfs:label ?valueLabel }} }}
  BIND(COALESCE(?valueLabel, STR(?value)) AS ?displayValue)
}}
ORDER BY STR(?predicate) STR(?displayValue)
"""


def relevant_chunks_query(paper_uri: str, question: str, limit: int) -> str:
    paper = sparql_iri(paper_uri)
    query = sparql_string(lucene_query_from_question(question))
    limit = max(1, int(limit))
    return f"""
PREFIX text: <http://jena.apache.org/text#>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX po: <http://www.essepuntato.it/2008/12/pattern#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?node ?score ?text ?nodeType ?sentence ?section ?sectionTitle ?sectionPosition ?sentencePosition WHERE {{
  GRAPH ?textGraph {{
    (?node ?score) text:query (rdfs:label {query} {limit}) .
    ?node rdfs:label ?text .
  }}
  GRAPH ?provGraph {{
    ?node prov:hadPrimarySource {paper} .
    OPTIONAL {{ ?node prov:wasDerivedFrom ?sentence }}
  }}
  OPTIONAL {{ GRAPH ?typeGraph {{ ?node rdf:type ?nodeType }} }}
  OPTIONAL {{
    GRAPH ?structureGraph {{
      {paper} po:contains ?section .
      ?section po:contains ?paragraph .
      ?paragraph po:contains ?sentence .
      OPTIONAL {{ ?section <{POSITION}> ?sectionPosition }}
      OPTIONAL {{ ?sentence <{POSITION}> ?sentencePosition }}
      OPTIONAL {{ ?section po:containsAsHeader ?header }}
    }}
    OPTIONAL {{ GRAPH ?headerLabelGraph {{ ?header rdfs:label ?sectionTitleLabel }} }}
    OPTIONAL {{ GRAPH ?headerValueGraph {{ ?header rdf:value ?sectionTitleValue }} }}
    BIND(COALESCE(?sectionTitleLabel, ?sectionTitleValue, STRAFTER(STR(?section), "/section/")) AS ?sectionTitle)
  }}
}}
ORDER BY DESC(?score) ?sectionPosition ?sentencePosition STR(?node)
LIMIT {limit}
"""


def all_paper_chunks_query(paper_uri: str) -> str:
    paper = sparql_iri(paper_uri)
    return f"""
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX po: <http://www.essepuntato.it/2008/12/pattern#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?node ?text ?nodeType ?sentence ?section ?sectionTitle ?sectionPosition ?sentencePosition WHERE {{
  GRAPH ?provGraph {{
    ?node prov:hadPrimarySource {paper} .
    OPTIONAL {{ ?node prov:wasDerivedFrom ?sentence }}
  }}
  GRAPH ?labelGraph {{
    ?node rdfs:label ?text .
  }}
  OPTIONAL {{ GRAPH ?typeGraph {{ ?node rdf:type ?nodeType }} }}
  OPTIONAL {{
    GRAPH ?structureGraph {{
      {paper} po:contains ?section .
      ?section po:contains ?paragraph .
      ?paragraph po:contains ?sentence .
      OPTIONAL {{ ?section <{POSITION}> ?sectionPosition }}
      OPTIONAL {{ ?sentence <{POSITION}> ?sentencePosition }}
      OPTIONAL {{ ?section po:containsAsHeader ?header }}
    }}
    OPTIONAL {{ GRAPH ?headerLabelGraph {{ ?header rdfs:label ?sectionTitleLabel }} }}
    OPTIONAL {{ GRAPH ?headerValueGraph {{ ?header rdf:value ?sectionTitleValue }} }}
    BIND(COALESCE(?sectionTitleLabel, ?sectionTitleValue, STRAFTER(STR(?section), "/section/")) AS ?sectionTitle)
  }}
}}
ORDER BY ?sectionPosition ?sentencePosition STR(?node)
"""


In [5]:
def rows_to_chunks(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    chunks_by_key: dict[tuple[str | None, str | None, str | None, str | None], dict[str, Any]] = {}
    for row in rows:
        node_uri = binding_value(row, "node")
        text = binding_value(row, "text")
        sentence_uri = binding_value(row, "sentence")
        section_uri = binding_value(row, "section")
        if not node_uri or not text:
            continue
        key = (node_uri, text, sentence_uri, section_uri)
        chunk = chunks_by_key.setdefault(
            key,
            {
                "node_uri": node_uri,
                "text": " ".join(text.split()),
                "sentence_uri": sentence_uri,
                "section_uri": section_uri,
                "section_title": binding_value(row, "sectionTitle"),
                "section_position": binding_int(row, "sectionPosition"),
                "sentence_position": binding_int(row, "sentencePosition"),
                "score": binding_float(row, "score"),
                "node_types": [],
            },
        )
        score = binding_float(row, "score")
        if score is not None:
            chunk["score"] = max(score, chunk["score"] or -math.inf)
        node_type = binding_value(row, "nodeType")
        if node_type and node_type not in chunk["node_types"]:
            chunk["node_types"].append(node_type)

    return list(chunks_by_key.values())


def lexical_score(question: str, text: str) -> float:
    terms = question_terms(question)
    words = re.findall(r"[A-Za-z0-9][A-Za-z0-9_-]+", text.lower())
    if not terms or not words:
        return 0.0

    word_set = set(words)
    matched_terms = set()
    overlap = 0.0
    for term in terms:
        prefix = term[:5]
        for word in word_set:
            if word == term or (len(prefix) >= 4 and word.startswith(prefix)):
                matched_terms.add(term)
                overlap += 1.0
                break

    coverage = len(matched_terms) / len(set(terms))
    return overlap + coverage


def chunk_answer_score(question: str, chunk: dict[str, Any]) -> float:
    text = chunk["text"]
    word_count = len(text.split())
    base = lexical_score(question, text)
    lucene = float(chunk.get("score") or 0.0)
    uri = str(chunk.get("node_uri") or "")
    substantive_bonus = 2.0 if word_count >= 8 else -1.0
    proposition_bonus = 1.5 if "/proposition/" in uri else 0.0
    concept_penalty = -1.0 if "/concept/" in uri and word_count < 8 else 0.0
    return base + math.log1p(max(lucene, 0.0)) + substantive_bonus + proposition_bonus + concept_penalty


def merge_chunks(*chunk_lists: list[dict[str, Any]]) -> list[dict[str, Any]]:
    merged: dict[tuple[str | None, str | None, str | None, str | None], dict[str, Any]] = {}
    for chunks in chunk_lists:
        for chunk in chunks:
            key = (
                chunk.get("node_uri"),
                chunk.get("text"),
                chunk.get("sentence_uri"),
                chunk.get("section_uri"),
            )
            existing = merged.get(key)
            if existing is None:
                merged[key] = dict(chunk)
                continue
            existing_score = existing.get("score") or 0.0
            new_score = chunk.get("score") or 0.0
            existing["score"] = max(existing_score, new_score)
    return list(merged.values())


def fetch_paper_metadata(paper_id: str) -> list[dict[str, str | None]]:
    paper_uri = paper_uri_from_id(paper_id)
    rows = run_select(paper_metadata_query(paper_uri))
    return [
        {
            "predicate": binding_value(row, "predicate"),
            "value": binding_value(row, "value"),
            "display_value": binding_value(row, "displayValue"),
        }
        for row in rows
    ]


def fetch_all_paper_chunks(paper_id: str) -> list[dict[str, Any]]:
    paper_uri = paper_uri_from_id(paper_id)
    return rows_to_chunks(run_select(all_paper_chunks_query(paper_uri)))


def retrieve_relevant_chunks(paper_id: str, question: str, limit: int = 12) -> list[dict[str, Any]]:
    paper_uri = paper_uri_from_id(paper_id)
    indexed_chunks: list[dict[str, Any]] = []
    if lucene_query_from_question(question):
        indexed_chunks = rows_to_chunks(run_select(relevant_chunks_query(paper_uri, question, limit=max(limit * 5, 40))))

    # Pull the paper-scoped evidence too, then rerank. This keeps the text index in the loop
    # but avoids short concept labels dominating answer generation.
    chunks = merge_chunks(indexed_chunks, fetch_all_paper_chunks(paper_id))
    for chunk in chunks:
        chunk["answer_score"] = chunk_answer_score(question, chunk)
    chunks.sort(key=lambda item: item.get("answer_score") or 0.0, reverse=True)
    return chunks[:limit]


## Answer Function

This notebook uses an extractive answer by default: it returns the highest-ranked paper-grounded evidence snippets and attributes them to section URIs. You can replace `build_extractive_answer` with an LLM call later while keeping the same retrieval and attribution layer.

In [6]:
def select_answer_chunks(chunks: list[dict[str, Any]], max_chunks: int = 4) -> list[dict[str, Any]]:
    substantive = [chunk for chunk in chunks if len(str(chunk.get("text", "")).split()) >= 8]
    candidates = substantive or chunks
    selected: list[dict[str, Any]] = []
    seen_text: set[str] = set()
    for chunk in candidates:
        text = str(chunk.get("text", "")).strip()
        if not text or text in seen_text:
            continue
        seen_text.add(text)
        selected.append(chunk)
        if len(selected) >= max_chunks:
            break
    return selected


def build_extractive_answer(question: str, chunks: list[dict[str, Any]], max_chunks: int = 4) -> str:
    selected = select_answer_chunks(chunks, max_chunks=max_chunks)
    if not selected:
        return "I could not find paper-grounded evidence for this question."
    return " ".join(str(chunk["text"]).strip() for chunk in selected)


def attribution_uris(chunks: list[dict[str, Any]], used_limit: int = 4) -> list[str]:
    uris: list[str] = []
    for chunk in chunks[:used_limit]:
        uri = chunk.get("section_uri") or chunk.get("sentence_uri") or chunk.get("node_uri")
        if isinstance(uri, str) and uri not in uris:
            uris.append(uri)
    return uris


def answer_question(paper_id: str, question: str, top_k: int = 12, answer_chunks: int = 4) -> dict[str, Any]:
    chunks = retrieve_relevant_chunks(paper_id, question, limit=top_k)
    used_chunks = select_answer_chunks(chunks, max_chunks=answer_chunks)
    answer = build_extractive_answer(question, used_chunks, max_chunks=answer_chunks)
    return {
        "question": question,
        "answer": answer,
        "attribute_uris": attribution_uris(used_chunks, used_limit=answer_chunks),
    }


## Try It

Change `PAPER_ID` and `QUESTION`, then run the cells below.

In [7]:
PAPER_ID = "learning_relation_entailment_with_structured_and_textual_information"
QUESTION = "What task does this paper introduce and why is it useful?"

result = answer_question(PAPER_ID, QUESTION)
result


{'question': 'What task does this paper introduce and why is it useful?',
 'answer': 'In this paper, we introduce a new task of predicting relation entailment: given two relations predicting whether one entails the other. We use these methods to establish several baselines for this task. Task Definition Let L denote a set of parent relations, given a set of N training child relations X train = {r i } N i=1 and their ground truth parents Y train = {r i } N i=1 (i.e., r i |= r i , r i ∈ L, i = 1, . . . , N This task can be conceptually interpreted as a multi-class classification problem, where parent relations r ∈ L are classes',
 'attribute_uris': ['https://w3id.org/twc/sudo/kg/section/8aef804d-e789-448f-a71d-84dabc426264',
  'https://w3id.org/twc/sudo/kg/section/716db4f9-6d03-46a5-b5d2-8086a96f8eb2']}

In [8]:
# Inspect the retrieved evidence when debugging answer quality.
retrieve_relevant_chunks(PAPER_ID, QUESTION, limit=5)


[{'node_uri': 'https://w3id.org/twc/sudo/kg/proposition/6801d9d7-b7b0-4416-9a5a-95006ae82e0a',
  'text': 'In this paper, we introduce a new task of predicting relation entailment: given two relations predicting whether one entails the other.',
  'sentence_uri': 'https://w3id.org/twc/sudo/kg/sentence/0e82699f-624f-4e4b-b19e-124b1fb41641',
  'section_uri': 'https://w3id.org/twc/sudo/kg/section/8aef804d-e789-448f-a71d-84dabc426264',
  'section_title': 'Introduction',
  'section_position': 1,
  'sentence_position': 1,
  'score': None,
  'node_types': ['https://w3id.org/twc/sudo/ontology#Argument',
   'https://w3id.org/twc/sudo/ontology#Idea'],
  'answer_score': 6.0},
 {'node_uri': 'https://w3id.org/twc/sudo/kg/proposition/8dce0285-6439-495c-9a58-666997f33867',
  'text': 'We use these methods to establish several baselines for this task.',
  'sentence_uri': 'https://w3id.org/twc/sudo/kg/sentence/5aac8735-64f8-4296-9120-b55ac0deac0c',
  'section_uri': 'https://w3id.org/twc/sudo/kg/section/8a